<a href="https://colab.research.google.com/github/csrsustain/HVAC-Optimization-/blob/main/FCU_Overnight_Operation_(Cooling).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# === INPUTS (from PEAK / TAB) ===
fcu_name = "............"     # Building and FCU number
airflow_ls = 200.0      # L/s   - airflow at overnight fan speed
sat = 14.0              # degC  - supply air temperature
rat = 22.0              # degC  - return (or room) air temperature
fan_kw = 0.09           # kW    - FCU fan electrical power
hours = 12.0            # h     - Retrieve this value from PEAK based on the total number of hours the FCU remained in fault.
nights = 365            # -     - nights per year

chiller_cop = 3.5       # Chiller efficiency
cost_per_kwh = 0.25     # GBP/kWh
co2_factor = 0.207      # kgCO2e/kWh

# === CONSTANTS ===
air_density = 1.204     # kg/m3
cp_air = 1.006          # kJ/kg.K

# === CALCULATION ===
delta_t = rat - sat
mass_flow = (airflow_ls / 1000.0) * air_density     # kg/s
kw_thermal = mass_flow * cp_air * delta_t           # kW cooling delivered
kw_chiller = kw_thermal / chiller_cop               # kW electrical, chiller
kw_total = kw_chiller + fan_kw                      # kW electrical, total

kwh_night = kw_total * hours
cost_night = kwh_night * cost_per_kwh
kwh_year = kwh_night * nights
cost_year = kwh_year * cost_per_kwh
co2_year = kwh_year * co2_factor

# === OUTPUT ===
W1, W2, W3 = 26, 14, 8      # label, value, unit column widths


def rule(left, mid, right, fill="\u2500"):
    print(left + fill * (W1 + 2) + mid + fill * (W2 + 2) + mid + fill * (W3 + 2) + right)


def row(label, value, unit=""):
    print(f"\u2502 {label:<{W1}} \u2502 {value:>{W2}} \u2502 {unit:<{W3}} \u2502")


def section(title):
    rule("\u251c", "\u253c", "\u2524")
    print(f"\u2502 {title:<{W1 + W2 + W3 + 6}} \u2502")
    rule("\u251c", "\u253c", "\u2524")


rule("\u250c", "\u252c", "\u2510")
print(f"\u2502 {'FCU OVERNIGHT COOLING COST - ' + fcu_name:<{W1 + W2 + W3 + 6}} \u2502")
print(f"\u2502 {'Sensible heat only (latent excluded)':<{W1 + W2 + W3 + 6}} \u2502")

section("INPUTS")
row("Airflow", f"{airflow_ls:,.1f}", "L/s")
row("Supply air temp (SAT)", f"{sat:,.1f}", "degC")
row("Return air temp (RAT)", f"{rat:,.1f}", "degC")
row("FCU fan power", f"{fan_kw:,.3f}", "kW")
row("Overnight hours", f"{hours:,.1f}", "h/night")
row("Nights per year", f"{nights:,}", "nights")
row("Chiller COP", f"{chiller_cop:,.2f}", "-")
row("Electricity tariff", f"{cost_per_kwh:,.3f}", "GBP/kWh")
row("CO2 factor", f"{co2_factor:,.3f}", "kg/kWh")

if delta_t <= 0:
    section("RESULT")
    row("Delta T (RAT - SAT)", f"{delta_t:,.2f}", "degC")
    row("Status", "NOT COOLING", "-")
    rule("\u2514", "\u2534", "\u2518")
    print("\nSAT >= RAT: the unit is not cooling, so there is no overnight cooling cost.")
else:
    section("LOAD")
    row("Delta T (RAT - SAT)", f"{delta_t:,.2f}", "degC")
    row("Mass flow", f"{mass_flow:,.3f}", "kg/s")
    row("Cooling delivered", f"{kw_thermal:,.2f}", "kW")

    section("ELECTRICAL DEMAND")
    row("Chiller draw", f"{kw_chiller:,.2f}", "kW")
    row("FCU fan draw", f"{fan_kw:,.2f}", "kW")
    row("Total draw", f"{kw_total:,.2f}", "kW")

    section("ENERGY, COST & CARBON")
    row("Energy per night", f"{kwh_night:,.2f}", "kWh")
    row("Cost per night", f"{cost_night:,.2f}", "GBP")
    row("Annual energy", f"{kwh_year:,.0f}", "kWh")
    row("Annual cost", f"{cost_year:,.2f}", "GBP")
    row("Annual CO2", f"{co2_year:,.0f}", "kgCO2e")
    rule("\u2514", "\u2534", "\u2518")
    print("\nThis is the avoidable cost if the FCU were switched off overnight.")

┌────────────────────────────┬────────────────┬──────────┐
│ FCU OVERNIGHT COOLING COST - FCU-01                    │
│ Sensible heat only (latent excluded)                   │
├────────────────────────────┼────────────────┼──────────┤
│ INPUTS                                                 │
├────────────────────────────┼────────────────┼──────────┤
│ Airflow                    │          200.0 │ L/s      │
│ Supply air temp (SAT)      │           14.0 │ degC     │
│ Return air temp (RAT)      │           22.0 │ degC     │
│ FCU fan power              │          0.090 │ kW       │
│ Overnight hours            │           12.0 │ h/night  │
│ Nights per year            │            365 │ nights   │
│ Chiller COP                │           3.50 │ -        │
│ Electricity tariff         │          0.250 │ GBP/kWh  │
│ CO2 factor                 │          0.207 │ kg/kWh   │
├────────────────────────────┼────────────────┼──────────┤
│ LOAD                                                  